# AfriDef — Full Experiment Runner (KSM 2026)

**Paper:** Adversarial-Resilient AI for Mobile Money Fraud Detection in Sub-Saharan Africa  
**Venue:** KSM 2026, Mogadishu, 02–05 August 2026  
**Runtime:** ~10–12 hr total on T4 GPU (5 seeds × full PaySim + IEEE-CIS)

### Instructions
1. Runtime → Change runtime type → **T4 GPU**
2. Run cells top-to-bottom
3. Results save to `/content/drive/MyDrive/afridef_results/`

In [ ]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/afridef_results', exist_ok=True)
print('Drive mounted.')

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────────────────
# PyG and friends for the T4 CUDA version on Colab (cu121)
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    else:
        print('OK:', cmd[:60])

run('pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.3.0+cu121.html')
run('pip install -q torch-geometric')
run('pip install -q stable-baselines3==2.4.0 gymnasium kaggle pyyaml scikit-learn')
print('All packages installed.')

In [ ]:
# ── Cell 3: Clone / pull the AfriDef repo ───────────────────────────────────
# Replace with your actual GitHub repo URL after pushing
REPO_URL = 'https://github.com/YOUR_USERNAME/afridef.git'  # <-- UPDATE THIS

if not os.path.exists('/content/afridef'):
    !git clone {REPO_URL} /content/afridef
else:
    !cd /content/afridef && git pull

%cd /content/afridef
!pip install -q -e .
print('Repo ready.')

In [ ]:
# ── Cell 4: Kaggle credentials ──────────────────────────────────────────────
import os, json
from google.colab import userdata

# Store your Kaggle token in Colab Secrets (key icon on left sidebar)
# Secret name: KAGGLE_TOKEN   value: your KGAT_xxx token
try:
    token = userdata.get('KAGGLE_TOKEN')
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/access_token'), 'w') as f:
        f.write(token)
    print('Kaggle token set from Colab Secrets.')
except Exception:
    print('Add KAGGLE_TOKEN to Colab Secrets (left sidebar → key icon).')

In [ ]:
# ── Cell 5: Download datasets ───────────────────────────────────────────────
os.makedirs('/content/afridef/data/raw', exist_ok=True)
%cd /content/afridef/data/raw

# PaySim1 (~500 MB)
if not os.path.exists('PS_20174392719_1491204439457_log.csv'):
    !kaggle datasets download -d ealaxi/paysim1 --unzip
    print('PaySim1 downloaded.')
else:
    print('PaySim1 already present.')

# IEEE-CIS (~200 MB) — accept rules at kaggle.com/competitions/ieee-fraud-detection first
if not os.path.exists('train_transaction.csv'):
    !kaggle competitions download -c ieee-fraud-detection --path .
    !unzip -q ieee-fraud-detection.zip
    print('IEEE-CIS downloaded.')
else:
    print('IEEE-CIS already present.')

%cd /content/afridef
!ls -lh data/raw/

In [ ]:
# ── Cell 6: Verify GPU and imports ──────────────────────────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

import torch_geometric, stable_baselines3 as sb3
print(f'PyG {torch_geometric.__version__} | SB3 {sb3.__version__} | PyTorch {torch.__version__}')

In [ ]:
# ── Cell 7: Run full AfriDef — PaySim, all 3 layers, 5 seeds ────────────────
# Full 6M rows, 5 seeds → results table for §6
# Estimated time: 4–6 hr on T4

import subprocess, json
from pathlib import Path

results = []

for seed in range(5):
    print(f'\n{'='*60}')
    print(f'SEED {seed}  — full PaySim (6.3M rows) — all layers')
    print('='*60)

    # Patch seed in config
    import yaml
    cfg_path = Path('configs/default.yaml')
    cfg = yaml.safe_load(cfg_path.read_text())
    cfg['seed'] = seed
    cfg_path.write_text(yaml.dump(cfg))

    cmd = [
        'python', 'scripts/baseline_graphsage.py',
        '--config', 'configs/default.yaml',
        '--nrows', '0',          # 0 = full dataset
        '--augment',
        '--trades',
        '--stackelberg',
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout[-3000:])
    if proc.returncode != 0:
        print('STDERR:', proc.stderr[-1000:])

    # Parse results from stdout
    lines = proc.stdout.split('\n')
    row = {'seed': seed}
    for line in lines:
        for key in ['AUROC', 'AP', 'F1']:
            if key in line and ':' in line:
                try:
                    row[key] = float(line.split(':')[-1].strip())
                except:
                    pass
    results.append(row)
    print(f'Seed {seed} done: {row}')

print('\nAll seeds complete.')

In [ ]:
# ── Cell 8: Compute mean ± std and save ─────────────────────────────────────
import numpy as np, pandas as pd, json

df_res = pd.DataFrame(results)
print('\nPer-seed results:')
print(df_res.to_string(index=False))

summary = {}
for col in ['AUROC', 'AP', 'F1']:
    if col in df_res.columns:
        vals = df_res[col].dropna()
        summary[col] = f'{vals.mean():.4f} ± {vals.std():.4f}'

print('\nSummary (mean ± std across 5 seeds):')
for k, v in summary.items():
    print(f'  {k:6s}: {v}')

# Save
out_path = '/content/drive/MyDrive/afridef_results/full_results.json'
with open(out_path, 'w') as f:
    json.dump({'per_seed': results, 'summary': summary}, f, indent=2)
df_res.to_csv('/content/drive/MyDrive/afridef_results/results_table.csv', index=False)
print(f'\nResults saved to Google Drive.')

In [ ]:
# ── Cell 9: Ablation table — run all 4 configurations ───────────────────────
# This builds Table 1: vanilla / +WGAN / +TRADES / full AfriDef

configs_to_run = [
    ('Vanilla GraphSAGE',          []),
    ('+ WGAN-GP (Layer 1)',         ['--augment']),
    ('+ TRADES (Layer 2)',          ['--trades']),
    ('AfriDef (Layers 1+2+3)',      ['--augment', '--trades', '--stackelberg']),
]

ablation = []
for name, flags in configs_to_run:
    print(f'\nRunning: {name}')
    cmd = ['python', 'scripts/baseline_graphsage.py',
           '--config', 'configs/default.yaml',
           '--nrows', '0'] + flags
    proc = subprocess.run(cmd, capture_output=True, text=True)
    row = {'Model': name}
    for line in proc.stdout.split('\n'):
        for key in ['AUROC', 'AP', 'F1']:
            if key in line and ':' in line:
                try:
                    row[key] = float(line.split(':')[-1].strip())
                except:
                    pass
    ablation.append(row)
    print(f'  Done: {row}')

df_ablation = pd.DataFrame(ablation)
print('\n=== TABLE 1 ===')
print(df_ablation.to_string(index=False))

df_ablation.to_csv('/content/drive/MyDrive/afridef_results/table1_ablation.csv', index=False)
print('Saved to Drive.')